# Indoor Scene Change Detection — Training & Evaluation Pipeline


## 1. Environment Setup


In [ ]:
!nvidia-smi


In [ ]:
!pip install ultralytics pandas numpy scipy scikit-learn matplotlib seaborn opencv-python -q
import os, json, shutil, glob, zipfile, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from PIL import Image as PILImage
from sklearn.metrics import classification_report, confusion_matrix
from scipy.optimize import linear_sum_assignment
from ultralytics import YOLO
from google.colab import files
from IPython.display import Image, display

print("Libraries ready.")


### 3a. Histogram Equalization Preprocessing (CLAHE)


In [ ]:
import cv2

def equalize_clahe(img_bgr, clip_limit=2.5, tile_grid_size=(8, 8)):

    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    l_eq = clahe.apply(l)
    lab_eq = cv2.merge((l_eq, a, b))
    return cv2.cvtColor(lab_eq, cv2.COLOR_LAB2BGR)

CLAHE_CLIP_LIMIT = 2.5
CLAHE_TILE_GRID = (8, 8)

def preprocess_dataset_histogram_eq(image_dir, clip_limit=CLAHE_CLIP_LIMIT,
                                     tile_grid_size=CLAHE_TILE_GRID, backup=True):

    paths = glob.glob(os.path.join(image_dir, '*.jpg')) + glob.glob(os.path.join(image_dir, '*.png'))
    if not paths:
        print(f"No images found in {image_dir} -- nothing to equalize.")
        return

    if backup:
        backup_dir = image_dir.rstrip('/') + '_original'
        if not os.path.exists(backup_dir):
            os.makedirs(backup_dir)
            for p in paths:
                shutil.copy2(p, os.path.join(backup_dir, os.path.basename(p)))
            print(f"Backed up {len(paths)} original images to {backup_dir}")
        else:
            print(f"Backup dir {backup_dir} already exists -- skipping backup "
                  f"(assuming equalization already ran once; re-running would double-apply CLAHE).")
            return

    n_ok, n_fail = 0, 0
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            n_fail += 1
            continue
        eq = equalize_clahe(img, clip_limit=clip_limit, tile_grid_size=tile_grid_size)
        cv2.imwrite(p, eq)
        n_ok += 1
    print(f"Histogram-equalized {n_ok} images in place in {image_dir} ({n_fail} unreadable/skipped).")

preprocess_dataset_histogram_eq('/content/custom_data/images')


## 3a-post. Exploratory Data Analysis — After Preprocessing (CLAHE Applied)

Same checks as above, re-run on the CLAHE-equalized images, plus a direct before/after comparison (using the automatic backup written to `images_original/` by `preprocess_dataset_histogram_eq`).

In [ ]:
print("="*60)
print("EDA (AFTER PREPROCESSING) -- CLAHE-equalized dataset")
print("="*60)

ORIG_BACKUP_DIR = RAW_IMAGE_DIR.rstrip('/') + '_original'

bright_after, contrast_after = [], []
for p in sample_paths:
    b, c = image_brightness_contrast(p)   # sample_paths now point at the equalized files
    if b is not None:
        bright_after.append(b); contrast_after.append(c)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(bright_vals, bins=25, alpha=0.5, label='Before CLAHE', color='#4C72B0')
axes[0].hist(bright_after, bins=25, alpha=0.5, label='After CLAHE', color='#C44E52')
axes[0].set_title('Brightness: Before vs After'); axes[0].legend()

axes[1].hist(contrast_vals, bins=25, alpha=0.5, label='Before CLAHE', color='#4C72B0')
axes[1].hist(contrast_after, bins=25, alpha=0.5, label='After CLAHE', color='#C44E52')
axes[1].set_title('Contrast: Before vs After'); axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Mean brightness  -> before: {np.mean(bright_vals):.1f}, after: {np.mean(bright_after):.1f}")
print(f"Mean contrast    -> before: {np.mean(contrast_vals):.1f}, after: {np.mean(contrast_after):.1f}")


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(10, 13))
compare_sample = rng.choice(sample_paths, size=3, replace=False)
for row, p in enumerate(compare_sample):
    fname = os.path.basename(p)
    orig_path = os.path.join(ORIG_BACKUP_DIR, fname)
    if not os.path.exists(orig_path):
        continue
    orig_img = cv2.cvtColor(cv2.imread(orig_path), cv2.COLOR_BGR2RGB)
    eq_img   = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2RGB)
    axes[row, 0].imshow(orig_img); axes[row, 0].set_title(f'Before: {fname}', fontsize=8); axes[row, 0].axis('off')
    axes[row, 1].imshow(eq_img);   axes[row, 1].set_title(f'After (CLAHE): {fname}', fontsize=8); axes[row, 1].axis('off')
plt.suptitle('Before vs After CLAHE -- Sample Comparison')
plt.tight_layout()
plt.show()
